# Traffic Sign Detection & Speed Limit Identification — Training & Evaluation

**SDAIA Academy · Computer Vision Systems Development · Final Project**

This notebook runs on **Google Colab** (`Runtime → Change runtime type → T4 GPU`).

| Step | What happens |
|---|---|
| 1. Setup | Install packages, mount Google Drive (so training survives a disconnect) |
| 2. Dataset | Download the Kaggle dataset, fix paths, explore class balance |
| 3. Training | Fine-tune a pretrained **YOLO11n** (transfer learning) with augmentation |
| 4. Evaluation report | Precision, Recall, mAP@0.5, mAP@0.5:0.95, per-class table, confusion matrix, PR curve |
| 5. Success & failure cases | Compare predictions to ground truth on test images |
| 6. Deployment | Export to **ONNX**, compare speed & accuracy vs PyTorch |
| 7. Save | Zip weights + results to put into the GitHub repo |

## 1. Setup

In [ ]:
!nvidia-smi -L
%pip install -q ultralytics kagglehub onnx onnxslim onnxruntime tabulate

In [ ]:
import os, time, shutil, random
from pathlib import Path

import cv2
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage
from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/traffic_signs')   # everything persistent lives here
RESULTS = PROJECT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

EPOCHS = 50
IMGSZ = 640
RUN_NAME = 'yolo11n_signs'
random.seed(0)

## 2. Dataset

**Source:** [Traffic Sign Detection Dataset (Kaggle)](https://www.kaggle.com/datasets/icebearogo/traffic-sign-detection-dataset) — already annotated in YOLO format
(one `.txt` per image: `class cx cy w h`, normalised 0–1).

If the download asks for credentials: create a Kaggle API token (kaggle.com → Settings → API),
then add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's **🔑 Secrets** panel.

In [ ]:
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass  # public dataset usually downloads without a token

import kagglehub
ds_root = Path(kagglehub.dataset_download('icebearogo/traffic-sign-detection-dataset'))
src_yaml = sorted(ds_root.rglob('data.yaml'))[0]
cfg = yaml.safe_load(open(src_yaml))
base = src_yaml.parent
print('dataset yaml:', src_yaml)

def find_split(key, folders):
    """Roboflow exports often ship with broken relative paths; locate the images folder ourselves."""
    candidates = [base / cfg[key]] if cfg.get(key) else []
    candidates += [base / f / 'images' for f in folders]
    for c in candidates:
        if c.is_dir():
            return str(c.resolve())
    return None

splits = {'train': find_split('train', ['train']),
          'val': find_split('val', ['valid', 'val']),
          'test': find_split('test', ['test'])}
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]

data_cfg = {k: v for k, v in splits.items() if v} | {'nc': len(names), 'names': names}
DATA_YAML = '/content/data.yaml'
yaml.safe_dump(data_cfg, open(DATA_YAML, 'w'), sort_keys=False)
EVAL_SPLIT = 'test' if splits['test'] else 'val'

print(yaml.safe_dump(data_cfg, sort_keys=False))

In [ ]:
def label_path(img_path):
    p = str(img_path)
    i = p.rfind(f'{os.sep}images{os.sep}')
    return Path(p[:i] + f'{os.sep}labels{os.sep}' + p[i + 8:]).with_suffix('.txt')

def list_images(split):
    return sorted(p for p in Path(splits[split]).rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})

counts, box_areas = {}, []
for split in [s for s in splits if splits[s]]:
    imgs = list_images(split)
    c = [0] * len(names)
    for img in imgs:
        lp = label_path(img)
        if lp.exists():
            for line in lp.read_text().split('\n'):
                if line.strip():
                    cls, _, _, w, h = map(float, line.split()[:5])
                    c[int(cls)] += 1
                    box_areas.append(w * h)
    counts[split] = c
    print(f'{split:5s}: {len(imgs):5d} images, {sum(c):5d} boxes')

counts_df = pd.DataFrame(counts, index=names)
counts_df

### Exploratory data analysis
Class balance matters: rare classes usually end up with the lowest mAP. Box size matters too —
small (far-away) signs are the hardest to detect.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
counts_df.plot.barh(ax=ax[0], stacked=True)
ax[0].set_title('Instances per class'); ax[0].set_xlabel('boxes')
ax[1].hist([a * 100 for a in box_areas], bins=50)
ax[1].set_title('Box size (% of image area)'); ax[1].set_xlabel('% of image'); ax[1].set_ylabel('boxes')
plt.tight_layout(); plt.savefig(RESULTS / 'dataset_eda.png', dpi=120); plt.show()
print(f'{sum(a < 0.01 for a in box_areas) / len(box_areas):.0%} of boxes cover < 1% of the image (small signs)')

In [ ]:
def draw_boxes(img, boxes, color, with_label=True):
    """boxes: list of (cls, [x1,y1,x2,y2] normalised, conf or None)"""
    h, w = img.shape[:2]
    for cls, (x1, y1, x2, y2), conf in boxes:
        p1, p2 = (int(x1 * w), int(y1 * h)), (int(x2 * w), int(y2 * h))
        cv2.rectangle(img, p1, p2, color, 2)
        if with_label:
            text = names[cls] + (f' {conf:.2f}' if conf is not None else '')
            cv2.putText(img, text, (p1[0], max(p1[1] - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img

def load_gt(img_path):
    lp, gt = label_path(img_path), []
    if lp.exists():
        for line in lp.read_text().split('\n'):
            if line.strip():
                cls, cx, cy, w, h = map(float, line.split()[:5])
                gt.append((int(cls), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], None))
    return gt

samples = random.sample(list_images('train'), 8)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, p in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    ax.imshow(draw_boxes(img, load_gt(p), (0, 255, 0))); ax.axis('off')
plt.suptitle('Training samples with ground-truth boxes'); plt.tight_layout(); plt.show()

## 3. Training — transfer learning with YOLO11n

* **Model:** YOLO11n pretrained on COCO (80 classes) → new head for our classes, all layers fine-tuned.
  Nano version: fast enough for real-time video on a CPU.
* **Preprocessing:** letterbox resize to 640×640, pixel values scaled to 0–1 (done by Ultralytics).
* **Augmentation:**
  * mosaic (4 images combined), random scale ±50 %, translate ±10 %
  * HSV colour jitter (lighting / weather changes)
  * **horizontal flip disabled (`fliplr=0`)** — a mirrored "30" is not a valid sign and would confuse digit reading.
* Checkpoints are saved to Google Drive; if Colab disconnects, just re-run all cells and training **resumes**.

In [ ]:
RUN_DIR = PROJECT / RUN_NAME
last = RUN_DIR / 'weights' / 'last.pt'

if last.exists():
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:          # raised when the run already finished
        print('Training already complete:', e)
else:
    YOLO('yolo11n.pt').train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=15, seed=0,
        project=str(PROJECT), name=RUN_NAME, exist_ok=True, plots=True,
        # augmentation
        mosaic=1.0, scale=0.5, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=0.0, fliplr=0.0,
    )

BEST = RUN_DIR / 'weights' / 'best.pt'
print('best weights:', BEST)

In [ ]:
display(IPImage(filename=str(RUN_DIR / 'results.png'), width=1100))

## 4. Evaluation report
Evaluated on the held-out **test** split (images never seen during training or model selection).

| Metric | Meaning |
|---|---|
| Precision | Of the signs the model reported, how many were correct |
| Recall | Of the real signs in the images, how many the model found |
| mAP@0.5 | Mean average precision, a box counts as correct at IoU ≥ 0.5 |
| mAP@0.5:0.95 | Stricter: averaged over IoU 0.5 → 0.95 (box must be tight) |

In [ ]:
model = YOLO(str(BEST))
m = model.val(data=DATA_YAML, split=EVAL_SPLIT, imgsz=IMGSZ, plots=True,
              project=str(PROJECT), name=f'eval_{EVAL_SPLIT}', exist_ok=True)

summary = pd.DataFrame({
    'metric': ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95', 'Inference (ms/img, GPU)'],
    'value': [m.box.mp, m.box.mr, m.box.map50, m.box.map, m.speed['inference']],
}).round(3)

per_class = pd.DataFrame([
    dict(zip(['class', 'precision', 'recall', 'mAP@0.5', 'mAP@0.5:0.95'], [names[c], *m.box.class_result(i)]))
    for i, c in enumerate(m.box.ap_class_index)
])
per_class.insert(1, 'instances', [counts_df.loc[n, EVAL_SPLIT] for n in per_class['class']])
per_class = per_class.sort_values('mAP@0.5').round(3).reset_index(drop=True)

display(Markdown('### Overall')); display(summary)
display(Markdown('### Per class (worst first)')); display(per_class)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(per_class['class'], per_class['mAP@0.5'])
ax.set_xlim(0, 1); ax.set_xlabel('mAP@0.5'); ax.set_title(f'Per-class mAP@0.5 ({EVAL_SPLIT} set)')
plt.tight_layout(); plt.savefig(RESULTS / 'per_class_map.png', dpi=120); plt.show()

for f in ['confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    p = Path(m.save_dir) / f
    if p.exists():
        display(Markdown(f'**{f}**')); display(IPImage(filename=str(p), width=900))
        shutil.copy(p, RESULTS / f)
shutil.copy(RUN_DIR / 'results.png', RESULTS / 'training_curves.png')

In [ ]:
report = f"""# Evaluation Report — YOLO11n traffic sign detector

Split: **{EVAL_SPLIT}** · image size {IMGSZ} · weights `best.pt`

## Overall
{summary.to_markdown(index=False)}

## Per class (worst first)
{per_class.to_markdown(index=False)}
"""
(RESULTS / 'eval_report.md').write_text(report)
summary.to_csv(RESULTS / 'metrics_summary.csv', index=False)
per_class.to_csv(RESULTS / 'metrics_per_class.csv', index=False)
print('saved', RESULTS / 'eval_report.md')

## 5. Success & failure cases
Each test image is compared to its ground truth: a prediction is correct when its box overlaps
a real sign with IoU ≥ 0.5 **and** has the same class. An image is a *success* when every sign is
found with the right class and there are no extra detections; otherwise it is a *failure*
(missed sign, wrong class, or false positive).

Green = ground truth · Red = prediction

In [ ]:
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def judge(gt, preds, thr=0.5):
    """Return list of problems (empty list = success)."""
    problems, used = [], set()
    for g_cls, g_box, _ in gt:
        best_j, best_iou = None, thr
        for j, (p_cls, p_box, _) in enumerate(preds):
            if j not in used and iou(g_box, p_box) >= best_iou:
                best_j, best_iou = j, iou(g_box, p_box)
        if best_j is None:
            problems.append(f'missed {names[g_cls]}')
        else:
            used.add(best_j)
            if preds[best_j][0] != g_cls:
                problems.append(f'{names[g_cls]} → predicted {names[preds[best_j][0]]}')
    problems += [f'false positive {names[c]} ({s:.2f})' for j, (c, _, s) in enumerate(preds) if j not in used]
    return problems

# self-check
assert abs(iou([0, 0, 2, 2], [1, 1, 3, 3]) - 1 / 7) < 1e-9 and iou([0, 0, 1, 1], [2, 2, 3, 3]) == 0
assert judge([(0, [0, 0, .5, .5], None)], [(0, [0, 0, .5, .5], .9)]) == []
assert judge([(0, [0, 0, .5, .5], None)], [(1, [0, 0, .5, .5], .9)]) == [f'{names[0]} → predicted {names[1]}']
assert judge([(0, [0, 0, .5, .5], None)], []) == [f'missed {names[0]}']

In [ ]:
successes, failures = [], []
for r in model.predict([str(p) for p in list_images(EVAL_SPLIT)], conf=0.25, imgsz=IMGSZ, stream=True, verbose=False):
    preds = [(int(c), b.tolist(), float(s)) for c, b, s in zip(r.boxes.cls, r.boxes.xyxyn, r.boxes.conf)]
    gt = load_gt(r.path)
    problems = judge(gt, preds)
    (failures if problems else successes).append((r.path, gt, preds, problems))

n = len(successes) + len(failures)
print(f'{len(successes)}/{n} test images fully correct ({len(successes) / n:.1%})')
kinds = pd.Series([p.split(' ')[0] if p.startswith(('missed', 'false')) else 'wrong class'
                   for *_, probs in failures for p in probs]).value_counts()
display(kinds.rename('failure type count').to_frame())

In [ ]:
def show_cases(cases, title, fname, k=4):
    cases = random.sample(cases, min(k, len(cases)))
    fig, axes = plt.subplots(1, len(cases), figsize=(5 * len(cases), 5), squeeze=False)
    for ax, (path, gt, preds, problems) in zip(axes[0], cases):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        img = draw_boxes(img, gt, (0, 255, 0), with_label=False)
        img = draw_boxes(img, preds, (255, 0, 0))
        ax.imshow(img); ax.axis('off')
        ax.set_title('\n'.join(problems[:3]) or 'all signs correct', fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(RESULTS / fname, dpi=120); plt.show()

# wrong-class mistakes first: the most interesting failures for a speed-limit reader
failures.sort(key=lambda f: not any('→' in p for p in f[3]))
show_cases(successes, 'Successful predictions', 'success_cases.png')
show_cases(failures[:12], 'Failure cases', 'failure_cases.png')

### Analysis & how the model could be improved
*Fill in after looking at the results above.* Typical findings for this task:
* **Similar speed limits get confused** (e.g. 30 ↔ 80, 50 ↔ 60) — the digits are only a few pixels wide.
  → higher input resolution (`imgsz=960`), or a second-stage classifier on the cropped sign.
* **Small / distant signs are missed** → larger model (YOLO11s/m), higher resolution, tiling.
* **Rare classes score lowest** (see per-class table) → collect more data or oversample them.
* **Domain gap** (dataset country vs Saudi roads, night, rain, glare) → fine-tune on local footage.

## 6. Deployment — ONNX export
ONNX makes the model portable: it runs with `onnxruntime` on any CPU without PyTorch,
which suits in-car or edge devices. We compare accuracy and CPU speed of both formats.

In [ ]:
ONNX = Path(model.export(format='onnx', imgsz=IMGSZ, simplify=True))

def cpu_ms_per_image(weights, imgs):
    mdl = YOLO(str(weights), task='detect')
    mdl.predict(str(imgs[0]), device='cpu', imgsz=IMGSZ, verbose=False)  # warm-up
    t = time.perf_counter()
    for p in imgs:
        mdl.predict(str(p), device='cpu', imgsz=IMGSZ, verbose=False)
    return (time.perf_counter() - t) / len(imgs) * 1000

bench_imgs = list_images(EVAL_SPLIT)[:50]
m_onnx = YOLO(str(ONNX), task='detect').val(data=DATA_YAML, split=EVAL_SPLIT, imgsz=IMGSZ, device='cpu',
                                          project=str(PROJECT), name='eval_onnx', exist_ok=True, plots=False)
deploy = pd.DataFrame([
    {'format': 'PyTorch (.pt)', 'size MB': BEST.stat().st_size / 1e6, 'mAP@0.5': m.box.map50,
     'CPU ms/img': cpu_ms_per_image(BEST, bench_imgs)},
    {'format': 'ONNX (.onnx)', 'size MB': ONNX.stat().st_size / 1e6, 'mAP@0.5': m_onnx.box.map50,
     'CPU ms/img': cpu_ms_per_image(ONNX, bench_imgs)},
]).round(3)
deploy['CPU FPS'] = (1000 / deploy['CPU ms/img']).round(1)
display(deploy)
with open(RESULTS / 'eval_report.md', 'a') as f:
    f.write(f'\n## Deployment (CPU, {len(bench_imgs)} images)\n{deploy.to_markdown(index=False)}\n')

## 7. Save artifacts
Download the zip, then in the repo put `best.pt` / `best.onnx` in `weights/`
and the contents of `results/` in `results/`.

In [ ]:
shutil.copy(BEST, PROJECT / 'best.pt')
shutil.copy(ONNX, PROJECT / 'best.onnx')
!cd "{PROJECT}" && zip -qr /content/traffic_signs_artifacts.zip best.pt best.onnx results
from google.colab import files
files.download('/content/traffic_signs_artifacts.zip')